## Setup

In [1]:
from pathlib import Path
import pandas as pd
import stringalign
from stringalign.evaluation import TranscriptionEvaluator

## Load data

In [2]:
data_path = Path("../data/")
dan_path = data_path / "raw" / "dan_correction.csv"
pylaia_path = data_path / "behandlet" / "all_lines.csv"

In [3]:
dan = pd.read_csv(dan_path, delimiter="|")
pylaia = pd.read_csv(pylaia_path)

# Ground truth uses \r\n for newline and dan_prediction uses only \n
dan["ground_truth"] = dan["ground_truth"].map(lambda s: s.replace("\r", ""))

## Set up functions to automatically detect replacements

In [4]:
def make_evaluator_from_row(row: pd.Series, references_name="ground_truth", predictions_name="dan_prediction") -> TranscriptionEvaluator:
    return TranscriptionEvaluator.from_strings(
        references=[row[references_name]],
        predictions=[row[predictions_name]],
    )

def get_aggregated_confusion_matrix(evaluator: TranscriptionEvaluator) -> stringalign.statistics.StringConfusionMatrix:
    return sum(
        (
            stringalign.statistics.StringConfusionMatrix.from_strings_and_alignment(
                reference=le.reference, predicted=le.predicted, alignment=le.alignment
            )
            for le in evaluator.line_errors
        ),
        start=stringalign.statistics.StringConfusionMatrix.get_empty(),
    )


def get_mistakes(confusion_matrix: stringalign.statistics.StringConfusionMatrix) -> dict[str, tuple[tuple[str, str], int]]:
    return tuple(
        ((op.generalize().substring, op.generalize().replacement), count)
        for op, count in confusion_matrix.edit_counts.most_common()
    )

def get_true_positives(confusion_matrix: stringalign.statistics.StringConfusionMatrix) -> dict[str, tuple[str, int]]:
    return tuple(confusion_matrix.true_positives.items())

def get_false_positives(confusion_matrix: stringalign.statistics.StringConfusionMatrix) -> dict[str, tuple[str, int]]:
    return tuple(confusion_matrix.false_positives.items())

def get_false_negatives(confusion_matrix: stringalign.statistics.StringConfusionMatrix) -> dict[str, tuple[str, int]]:
    return tuple(confusion_matrix.false_negatives.items())

## Compute replacements

We use optimal string alignment to automatically detect character replacements. For example, if we have the two strings:

`string`  
`align`

where `string` is the "ground truth" and `align` is the "prediction. An optimal alignment of these two strings could be

`string-`  
`al-i-gn`

which is equivalent to the following operations:

* `Replace('a', 's')`
* `Replace('l', 't')`
* `Insert('r')`
* `Keep('i')`
* `Insert('n')`
* `Keep('g')`
* `Delete('n')`

From this, we can compute statistics on how common different character replacements are. However, the strings `string` and `align` have other alignments that are just as valid as well. Take for example:

`string-`  
`-ali-gn`

* `Insert('s')`
* `Replace('a', 't')`
* `Replace('l', 'r')`
* `Keep('i')`
* `Insert('n')`
* `Keep('g')`
* `Delete('n')`

To mitigate this somewhat, we can aggregate the replacements:

* `Replace('al', 'str')`
* `Keep('i')`
* `Insert('n')`
* `Keep('g')`
* `Delete('n')`

This will give much more stable alignments (but there may still be some strings that have multiple possible alignments, particularly when letters swap places.

## Types of error statistics:

We compute four types of error statistics: "mistakes", "true positives", "false positives" and "false negatives".

### Mistakes
A mistake, `('e', 'a'), 3`, should be interpreted as there were three times the letter 'a' was transcribed as the letter 'e'.

### True positives
A true positive, `'a', 10`, means that there were ten times the letter 'a' was correctly transcribed

### False positives
A false positive, ('a', 2) means that there were two times the letter 'a' was transcribed when something else should have been transcribed

### False negatives
A false negatives, ('a', 5) means that there were five times the letter 'a' should have been transcribed, but something else was transcribed.

## Example
The alignment operations `[Replace('al', 'str'), Keep('i'), Insert('n'), Keep('g'), Delete('n')` have the following error statistics:

* **Mistakes**: `('al', 'str'), 1`, `('n', ''), 1`, `('', 'n')`
* **True positives**: `('i', 1)`, `('g', 1)`
* **False positives**: `('al', 1)`, `('n', 1)`
* **False negatives**: `('str', 1)`, `('n', 1)`

In [5]:
dfs = {"dan": dan, "pylaia": pylaia}
aggregated_confusion_matrices = {}

for model, df in dfs.items():
    df["evaluators"] = df.apply(make_evaluator_from_row, axis=1, predictions_name=f"{model}_prediction")
    df["aggregated_confusion_matrices"] = df["evaluators"].map(get_aggregated_confusion_matrix)
    
    df["mistakes"] = df["aggregated_confusion_matrices"].map(get_mistakes)
    df["true_positives"] = df["aggregated_confusion_matrices"].map(get_true_positives)
    df["false_positives"] = df["aggregated_confusion_matrices"].map(get_false_positives)
    df["false_negatives"] = df["aggregated_confusion_matrices"].map(get_false_negatives)
    
    aggregated_confusion_matrices[model] = sum(
        (cm for cm in df["aggregated_confusion_matrices"]),
        start=stringalign.statistics.StringConfusionMatrix.get_empty(),
    )

## Look for hallucinations
(By finding contiguous errors longer than half the text size)

### DAN

In [6]:
for document in dan.itertuples():
    if any(len(mistake[0][0]) > 0.5 * len(document.ground_truth) for mistake in document.mistakes):
        print("===")
        print("url")
        print("---")
        print(document.arkindex_page_url)
        print("\ntruth")
        print("-----")
        print(document.ground_truth)
        print("\nprediction")
        print("----------")
        print(document.dan_prediction)
        print("\nmistakes")
        print("--------")
        print(document.mistakes)

===
url
---
https://demo.arkindex.org/element/45c6ff3a-ce7a-4e0f-98d8-2378fb937f55

truth
-----
Ivar Aasen.
Theatergaden.
Kristiania.

prediction
----------
S. D. Breve det er det et skal skal skrive at jeg skal se det ikke saa skal skrive til at see det er det et skal skrive til at see det en store det er det skal se det en store det er saa skal jeg skal se det e det en store det er saa saa saa saa saa saa saa saa saa saa saa saa saa saa det er saa saa saa saa saa saa saa saa saa det var saa saa saa saa det var saa saa saa saa saa det saa saa saa saa saa saa saa saa saa saa det er saa saa saa saa saa saa saa det er det er det er det er det er det er det en saa saa saa saa saa saa saa saa saa saa saa saa saa saa saa saa saa saa saa saa saa det var saa saa saa saa det var saa saa saa saa det var saa saa saa saa det var saa saa saa saa det saa saa saa saa det er det saa saa saa det er det saa saa saa saa det er det saa saa saa saa saa det er saa saa saa saa det er saa saa saa saa saa det

### PyLaia

In [7]:
for document in pylaia.itertuples():
    if any(len(mistake[0][0]) > 0.5 * len(document.ground_truth) for mistake in document.mistakes):
        print("===")
        print("url")
        print("---")
        print(document.arkindex_page_url)
        print("\ntruth")
        print("-----")
        print(document.ground_truth)
        print("\nprediction")
        print("----------")
        print(document.pylaia_prediction)
        print("\nmistakes")
        print("--------")
        print(document.mistakes)

===
url
---
https://demo.arkindex.org/element/3605481a-f672-465d-96b4-8b5adbf9608a

truth
-----
No.

prediction
----------
20.

mistakes
--------
((('20', 'No'), 1),)


## Row-wise data

### DAN

In [8]:
df = dan[["ground_truth", "dan_prediction", "arkindex_page_url", "mistakes", "true_positives", "false_positives", "false_negatives"]]
df.to_csv("dan_errors.csv", index=False)

df

,ground_truth,dan_prediction,arkindex_page_url,mistakes,true_positives,false_positives,false_negatives
0,Rosenvænget 28 - nov - 1910\nHr Direktør J. Th...,Rosenværget 28-nor-1910\nHr Direktør Ø. Thiis!...,https://demo.arkindex.org/element/02f02f39-fd1...,"(((, ), 3), ((r, n), 1), ((r, v ), 1), ((Ø, J...","((R, 2), (o, 17), (s, 25), (e, 76), (n, 29), (...","((r, 4), (Ø, 1), (B, 2), (K, 1))","((n, 1), ( , 4), (v, 2), (J, 2), (s, 1), (p, 1..."
1,"Eivind Astrup,\nChristiania.\n21/12-95.\nTil V...",21/12-55.\nTil Verdens Gangs Redaktion:\nTilla...,https://demo.arkindex.org/element/0ad7f5ba-d7e...,"(((h, b), 2), ((, Eivind Astrup,\nChristiania....","((2, 2), (1, 2), (/, 1), (-, 1), (5, 1), (., 2...","((5, 1), (o, 2), (m, 2), (h, 4), (a, 1), (f, 1...","((E, 2), (i, 5), (v, 1), (n, 4), (d, 1), ( , 1..."
2,Flensburg 16.11.13\nSelvfølglig ikke. -\nAlt g...,Hensbarg 16. 11.13\nSerfølglig ikke. -\nAlt gi...,https://demo.arkindex.org/element/0aeebaca-b2e...,"(((a, u), 2), ((l, t), 2), ((n, t), 2), ((H, F...","((e, 22), (n, 9), (s, 7), (b, 2), (r, 12), (g,...","((H, 1), (a, 2), ( , 1), (r, 4), (n, 4), (l, 5...","((F, 1), (l, 7), (u, 3), (v, 2), (a, 2), (t, 7..."
3,Fru Camilla Collett.\nUndertegnede bringer sin...,Fru Camilla Collett.\nUndertegnede bringer sin...,https://demo.arkindex.org/element/102ca72a-6c6...,"(((,, ), 1), ((e, ), 1), ((s, ), 1), ((ig, y),...","((F, 2), (r, 23), (u, 3), ( , 41), (C, 2), (a,...","((,, 1), (e, 1), (s, 1), (i, 1), (g, 1), (D, 1...","((y, 1), (G, 2), (u, 1), (s, 2), (t, 1), (a, 1..."
4,av brev Frøis Frøisland 12/10-25\nvar forma sl...,an brev trois brøisland i 11/10-25\n- - - var ...,https://demo.arkindex.org/element/124d818b-d57...,"(((o, e), 5), ((, e), 4), ((i, e), 4), ((e, ),...","((a, 13), ( , 72), (b, 1), (r, 23), (e, 25), (...","((n, 5), (t, 5), (o, 7), (b, 1), ( , 9), (i, 1...","((v, 5), (F, 2), (ø, 2), (2, 1), (a, 6), (s, 7..."
...,...,...,...,...,...,...,...
85,Kristiania 8/2 - XIII\nIllustrissime!\nVed nær...,Kristiania 6/4-77\nHustrissime!\nHustrissime!\...,https://demo.arkindex.org/element/51cea8b8-41d...,"(((6, 8), 1), ((4, 2 ), 1), ((77, XIII), 1), ...","((K, 1), (r, 18), (i, 10), (s, 9), (t, 6), (a,...","((6, 1), (4, 1), (7, 2), (H, 2), (u, 1), (s, 4...","((8, 1), (2, 1), ( , 4), (X, 1), (I, 4), (l, 2..."
86,videnskaben.\nHvis under-\ntegnede til denne t...,videnskaben.\nHvis under-\nhegnede til denne t...,https://demo.arkindex.org/element/51cea8b8-41d...,"(((h, t), 1), ((fsa, på), 1), (((A, M), 1), ((...","((v, 5), (i, 13), (d, 12), (e, 24), (n, 13), (...","((h, 1), (f, 1), (s, 1), (a, 1), ((, 1), (A, 1...","((t, 2), (p, 1), (å, 1), (M, 1), (A, 1), (l, 1..."
87,Kristiania 23 januar 1893.\nFru Camilla Collet...,Kristiania 23 januar 1893.\nFru Camilla Collet...,https://demo.arkindex.org/element/9cabe071-9dd...,"(((., ), 1), ((- , ), 1))","((K, 2), (r, 14), (i, 18), (s, 15), (t, 14), (...","((., 1), (-, 1), ( , 1))",()
88,"S. T:\nHr. Ivar Aasen.\nI Haab om, at De undsk...","S. T:\nHr. Svar Aasen.\nI Haabom, at De undsky...",https://demo.arkindex.org/element/a717b89b-055...,"(((, ""), 4), ((;, ,), 2), ((S, I), 1), ((, ),...","((S, 3), (., 7), ( , 99), (T, 1), (:, 2), (\n,...","((S, 1), (f, 1), (n, 1), (\n, 1), (u, 1), (e, ...","((I, 2), ( , 6), (J, 1), (r, 6), (., 3), ("", 5..."


### PyLaia

In [9]:
df = pylaia[["ground_truth", "pylaia_prediction", "arkindex_page_url", "mistakes", "true_positives", "false_positives", "false_negatives"]]
df.to_csv("dan_errors.csv", index=False)

df

,ground_truth,pylaia_prediction,arkindex_page_url,mistakes,true_positives,false_positives,false_negatives
0,Rosenvænget 28 - nov - 1910,Rosenværget 28- nov-1910,https://demo.arkindex.org/element/02f02f39-fd1...,"(((, ), 3), ((r, n), 1))","((R, 1), (o, 2), (s, 1), (e, 2), (n, 2), (v, 2...","((r, 1),)","((n, 1), ( , 3))"
1,Hr Direktør J. Thiis!,Hr Direktør J. Thiis!,https://demo.arkindex.org/element/02f02f39-fd1...,(),"((H, 1), (r, 3), ( , 3), (D, 1), (i, 3), (e, 1...",(),()
2,"De ved naturligvis, at jeg er bleven","De ved naturligvis, at jeg er bliven",https://demo.arkindex.org/element/02f02f39-fd1...,"(((i, e), 1),)","((D, 1), (e, 5), ( , 6), (v, 3), (d, 1), (n, 2...","((i, 1),)","((e, 1),)"
3,"Ridder af St. Olavsordenen, og","Ridder af St. Olavsordenen, og",https://demo.arkindex.org/element/02f02f39-fd1...,(),"((R, 1), (i, 1), (d, 3), (e, 3), (r, 2), ( , 4...",(),()
4,da De vist har været med til at for-,da De vist har været med til at for-,https://demo.arkindex.org/element/02f02f39-fd1...,(),"((d, 2), (a, 3), ( , 8), (D, 1), (e, 3), (v, 2...",(),()
...,...,...,...,...,...,...,...
1559,"er forfærdelig, og dog vil jeg","at forfærdelig, og dog vil jeg",https://demo.arkindex.org/element/fe817b1f-c47...,"(((at, er), 1),)","(( , 5), (f, 2), (o, 3), (r, 2), (æ, 1), (d, 2...","((a, 1), (t, 1))","((e, 1), (r, 1))"
1560,kun Ensomheden. Gud,kun Ensomheden. Gud,https://demo.arkindex.org/element/fe817b1f-c47...,(),"((k, 1), (u, 2), (n, 3), ( , 2), (E, 1), (s, 1...",(),()
1561,hjelpe Deres hengivne,hjelpe Deres hengivne,https://demo.arkindex.org/element/fe817b1f-c47...,(),"((h, 2), (j, 1), (e, 6), (l, 1), (p, 1), ( , 2...",(),()
1562,Agnes Kjerulf,Syns Kjerulf,https://demo.arkindex.org/element/fe817b1f-c47...,"(((Sy, Ag), 1), ((, e), 1))","((n, 1), (s, 1), ( , 1), (K, 1), (j, 1), (e, 1...","((S, 1), (y, 1))","((A, 1), (g, 1), (e, 1))"


## Most common edits overall (aggregated)

### DAN

In [10]:
for op, count in aggregated_confusion_matrices["dan"].edit_counts.most_common(50):
    print(op, count)

Insert(substring=' ') 51
Replace(substring='e', replacement='a') 35
Insert(substring='e') 28
Replace(substring='i', replacement='e') 28
Insert(substring='"') 27
Replace(substring='e', replacement='i') 25
Replace(substring='h', replacement='k') 25
Replace(substring='a', replacement='o') 24
Insert(substring='r') 23
Replace(substring='l', replacement='t') 22
Delete(substring=',') 22
Replace(substring='r', replacement='s') 22
Delete(substring='e') 19
Insert(substring='.') 16
Delete(substring='.') 16
Delete(substring='r') 15
Replace(substring='o', replacement='a') 15
Replace(substring='v', replacement='r') 15
Insert(substring='s') 14
Replace(substring='n', replacement='r') 14
Delete(substring='n') 13
Insert(substring=',') 13
Delete(substring=' ') 12
Replace(substring='s', replacement='r') 12
Replace(substring='o', replacement='e') 11
Replace(substring='D', replacement='d') 11
Replace(substring='t', replacement='l') 10
Replace(substring='a', replacement='e') 10
Insert(substring='n') 10
Repla

### PyLaia

In [11]:
for op, count in aggregated_confusion_matrices["pylaia"].edit_counts.most_common(50):
    print(op, count)

Insert(substring='"') 82
Insert(substring=' ') 40
Insert(substring=',') 34
Insert(substring='e') 33
Insert(substring='r') 32
Insert(substring='s') 31
Insert(substring='.') 31
Delete(substring='.') 28
Delete(substring=' ') 27
Insert(substring='n') 20
Replace(substring='e', replacement='a') 20
Replace(substring='a', replacement='o') 19
Replace(substring='e', replacement='i') 19
Replace(substring='i', replacement='e') 18
Insert(substring='-') 16
Replace(substring='r', replacement='s') 15
Replace(substring='t', replacement='l') 14
Insert(substring='i') 13
Delete(substring='r') 13
Replace(substring='d', replacement='D') 12
Replace(substring='l', replacement='t') 11
Replace(substring='o', replacement='e') 11
Replace(substring='h', replacement='k') 11
Replace(substring='g', replacement='j') 10
Delete(substring=',') 10
Replace(substring='m', replacement='n') 9
Replace(substring='e', replacement='s') 9
Delete(substring='e') 9
Replace(substring='D', replacement='d') 9
Replace(substring='.', repl

## Observations from the character errors

There are a lot of spaces missing in the transcription (`Insert(substring=' ')` means that a space should be added to the transcription to make it equal to the ground truth). However, it's very difficult to automatically detect whether the whitespace is meaningful or not (e.g. `Rosenvænget 28 - nov - 1910` versus `Rosenvænget 28-nov-1910`). 

We also see that out of the 10 most common replacements for both models, seven are shared. For example, the most common replacement is the letter `'e'` being transcribed as an `'a'`.

## Save aggregated errors

In [13]:
import json
for model, confusion_matrix in aggregated_confusion_matrices.items():
    data = {
        "mistakes": get_mistakes(confusion_matrix),
        "true_positives": get_true_positives(confusion_matrix),
        "false_positives": get_false_positives(confusion_matrix),
        "false_negatives": get_false_negatives(confusion_matrix),
    }
    Path(f"aggregated_errors_{model}.json").write_text(json.dumps(data))